# EP06｜機率：分佈、條件機率、最大概似估計

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep06/ep06_probability.ipynb)

這份 notebook 對應網站上的 [EP06 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep06/)，段落和網站一樣，可以邊讀邊跑。EP00 看到，模型每一步替詞彙表裡的每個 token 給一個機率：「今天天氣很」後面，「冷」29.3%、「熱」22.9%⋯⋯全部加起來 100%，再依機率抽一個接上去。前面幾集留下三個問題：「機率分佈」正式是什麼？一句話的機率為什麼是每一步的機率連乘（EP02）？模型的參數怎樣才算好（EP05 的預告：讓訓練資料的機率越大越好）？答案是**條件機率**、**機率的鏈鎖律**和**最大概似估計**：一句話的機率，是每個 token「給定前面全部」的條件機率連乘；訓練語言模型，就是找讓訓練資料的機率最大的參數。

1. 工具一：機率與分佈（含丟硬幣、骰子與下一個 token 的圖）
2. 工具二：聯合機率與條件機率（含聯合機率表與條件機率表的圖）
3. 工具三：機率的鏈鎖律（含一句話每一步條件機率的圖）
4. 工具四：最大概似估計（概似函數的圖在第 6 節動手做畫）
5. **本集重點：訓練語言模型＝最大概似估計**（含玩具模型往上爬的圖）
6. 動手做：從零寫出 `empirical_distribution`、`conditional`、`chain_log_prob`、`bernoulli_mle`、`log_likelihood` 並用檢查函式測試；寫出畫概似函數的 `draw_likelihood`、`draw_log_likelihood`；和 PyTorch 對照
7. 弄壞它：條件機率除以錯的總和、把 token 當成彼此獨立、連乘下溢、資料太少的最大概似估計、沒有固定 seed

最後是參考解答。

這份 notebook 只用到 NumPy 和 matplotlib，Colab 已經內建，不用安裝任何東西；前幾集寫好的 `softmax`（EP01）、`log_softmax`、`sequence_logprob`（EP02）、`numerical_derivative`、`numerical_gradient`（EP04）也已經定義在下面，不用匯入。在自己的電腦上，用 EP01 建立的 `llm-labs` 專案開啟即可。動手做的最後一個檢查和「和 PyTorch 對照」那一格會和 PyTorch 比對：Colab 已經內建 PyTorch；沒裝 PyTorch 的話（EP01 步驟 7），那些地方會自動跳過。

**使用方式**：從上到下依序執行（Shift + Enter）。標著「先猜猜看」的地方，先在心裡猜結果再執行。參考解答在最後，先自己寫寫看。

In [ ]:
# Colab 已經內建 NumPy 和 matplotlib，不用安裝；在自己的電腦上，請用 EP01 建立的 llm-labs 環境。
# %pip install -q numpy matplotlib

import math
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| matplotlib", matplotlib.__version__)

這份 notebook 用 NumPy 2.5 測試過。程式格裡的註解是說明，印出來的結果就在格子下方。用到亂數的格子都固定了 seed，結果會和網站上一模一樣（弄壞它 5 除外）。圖上的文字用英文，因為 matplotlib 預設的字型沒有中文，會變成方框。

下面是前幾集動手做寫好的函式，這集會用到：EP01 的 `softmax`；EP02 的 `logsumexp`、`log_softmax`（不先算 Softmax 的 log 機率）和 `sequence_logprob`（一串機率的 Σ log，不先連乘）；EP04 的 `numerical_derivative`（中央差分，$h$ 預設 `1e-5`）和 `numerical_gradient`。`nd` 是 `numerical_derivative` 的短名字，和網站上的 `from ep04.calculus import numerical_derivative as nd` 一樣。最後的 `tidy` 是畫圖用的小工具：淡灰色格線、拿掉上面和右邊的框。

**寫法約定**：$P(\ldots)$ 念「⋯⋯的機率」（EP02）；丟硬幣時，1 代表正面、0 代表反面。

In [ ]:
def softmax(z, axis=-1):                  # EP01
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)   # 每一列減掉自己的最大值
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)      # 每一列除以自己的總和


def logsumexp(z, axis=-1, keepdims=False):    # EP02
    """數值穩定地算 log(Σ exp(z))，沿著 axis 做。z 可以是任意形狀。
    回傳形狀和 z.sum(axis=axis, keepdims=keepdims) 一樣：
    z 形狀 (4, 7)、axis=-1 → (4,)；keepdims=True → (4, 1)。"""
    z = np.asarray(z, dtype=float)
    m = z.max(axis=axis, keepdims=True)                              # 每一列自己的最大值
    out = m + np.log(np.sum(np.exp(z - m), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)                             # 拿掉長度 1 的那個維度
    return out


def log_softmax(z, axis=-1):              # EP02
    """回傳 log(softmax(z))，但不能先算 softmax 再取 log。輸出形狀和 z 一樣。"""
    z = np.asarray(z, dtype=float)
    return z - logsumexp(z, axis=axis, keepdims=True)                # keepdims=True 才能廣播回每一列


def sequence_logprob(probs):              # EP02
    """一串機率 p_1 … p_n，回傳 log(p_1 × … × p_n)，但不能先連乘。
    probs 形狀 (..., n)，回傳形狀 (...)：一維輸入就回傳一個數。"""
    return np.sum(np.log(probs), axis=-1)


def numerical_derivative(f, x, h=1e-5):   # EP04
    """用中央差分 (f(x + h) - f(x - h)) / (2h) 近似 f 在 x 的導數 f'(x)。
    f 吃一個數、吐一個數；x 是一個數。回傳一個數。"""
    return (f(x + h) - f(x - h)) / (2 * h)


def numerical_gradient(f, x, h=1e-5):     # EP04
    """f 吃一個 NumPy 陣列、吐一個數。對 x 的每一格各做一次中央差分，
    回傳和 x 形狀一樣的陣列（每一格是 f 對那一格的偏導數）。不能改到呼叫端的 x。"""
    x = np.array(x, dtype=float)          # 複製一份、轉成小數：不改到呼叫端，整數輸入也能動一點點
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):       # (0,)、(1,)…；二維時是 (0, 0)、(0, 1)…
        old = x[idx]
        x[idx] = old + h
        f_plus = f(x)
        x[idx] = old - h
        f_minus = f(x)
        x[idx] = old                      # 復原：直接放回原值，不靠 +h、-h 抵消
        grad[idx] = (f_plus - f_minus) / (2 * h)
    return grad


nd = numerical_derivative                 # 短名字


def tidy(ax):                             # 畫圖用：淡灰色格線、拿掉上面和右邊的框
    ax.grid(True, color="0.88", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

## 1. 工具一：機率與分佈

### 樣本空間、事件、機率

丟一顆骰子，所有可能的結果 $\{1, 2, 3, 4, 5, 6\}$ 叫**樣本空間 sample space**；我們關心的一組結果叫**事件 event**，例如「丟出偶數」是 $\{2, 4, 6\}$。**機率 probability** 替每個事件配一個數，遵守三個規則：

1. 每個事件的機率在 0 到 1 之間：0 是（照這個機率）不會發生，1 是一定發生。
2. 整個樣本空間的機率是 1：一定會丟出某一面。
3. 不會同時發生（**互斥 mutually exclusive**）的事件，機率可以相加：$P(\{2, 4, 6\}) = P(2) + P(4) + P(6)$。

公平的骰子每一面是 $\frac{1}{6}$，所以 $P(\text{偶數}) = \frac{3}{6} = 0.5$。EP00 的模型也一樣：樣本空間是詞彙表裡的 151,936 個 token，全部的機率加起來是 1（規則 2）；「下一個是『冷』或『熱』」是一個事件，機率是 29.3% + 22.9% = 52.2%（規則 3）。

### 丟硬幣：次數越多，比例越靠近機率

「正面的機率是 0.5」是什麼意思？丟很多次，正面的比例會靠近 0.5。用 EP01 的固定 seed 模擬，`rng.integers(0, 2)` 隨機給 0 或 1。第一行是前 10 次的結果，之後每一行是 $n$ 和前 $n$ 次裡正面的比例。**先猜猜看**：丟 20 次會比丟 10 次更靠近 0.5 嗎？

In [ ]:
rng = np.random.default_rng(0)
flips_sim = rng.integers(0, 2, size=1_000_000)    # 丟一百萬次：1 是正面、0 是反面
print(flips_sim[:10])
for n in [10, 20, 100, 1000, 10_000, 100_000, 1_000_000]:
    print(n, flips_sim[:n].mean())                # 前 n 次裡正面的比例

0/1 陣列的平均就是 1 的比例。10 次是 0.4，20 次反而跳到 0.7（第 11 到 20 次剛好全是正面）：不是每多丟一次就更近，但次數越多，越穩定地靠近 0.5。這叫**大數法則 law of large numbers**。反過來，**手上只有資料時，就用比例來猜機率**；工具四會看到，這正是最大概似估計的答案。

把每一個 $n$ 的比例都畫出來（橫軸是對數刻度）。`np.cumsum` 是「加到目前為止」：第 $n$ 格是前 $n$ 次的正面數，除以 $n$ 就是比例；圓點是上面印出的那幾個 $n$。

In [ ]:
n_all = np.arange(1, flips_sim.size + 1)                 # 1、2、⋯、1000000
running = np.cumsum(flips_sim) / n_all                   # 前 n 次裡正面的比例
marks = [10, 20, 100, 1000, 10_000, 100_000, 1_000_000]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(n_all, running, color="C0", linewidth=1.6, label="proportion of heads so far")
ax.axhline(0.5, color="C3", linestyle="--", linewidth=1.2, label="probability 0.5")
ax.plot(marks, running[np.array(marks) - 1], "o", color="C0", markersize=5)
for m, off in [(10, (10, -4)), (20, (0, 9)), (100, (0, 9)), (1000, (0, 9))]:   # 前幾個點寫出數字
    ax.annotate(f"{running[m - 1]:g}", (m, running[m - 1]), xytext=off, textcoords="offset points",
                ha="left" if off[0] else "center", fontsize=9, color="0.25")
ax.set(xscale="log", xlim=(1, 1.2e6), ylim=(0, 1.05), xlabel="number of flips n (log scale)",
       ylabel="proportion of heads", title="proportion of heads in the first n flips (seed 0)")
ax.legend(loc="lower right", frameon=False)
tidy(ax)
plt.show()

前幾十次在 0.33 到 1 之間大幅跳動（10 次 0.4、20 次 0.7），之後越來越貼近 0.5 的紅色虛線。

（網站上有互動元件：拖動丟的次數、換 seed，看正面的比例和它離 0.5 的距離。）

### 隨機變數與分佈

把結果寫成一個數，叫**隨機變數 random variable**，習慣用大寫字母，例如 $X$＝骰子的點數，$P(X = 3) = \frac{1}{6}$。把每個值的機率都列出來，就是**機率分佈 probability distribution**。這集只講**離散 discrete** 的分佈：值可以一個一個列出來。EP05 的 `rng.normal` 抽出的數可以是任何小數，是**連續 continuous** 的分佈（和函數「連續」不是同一件事），EP13 講初始化時再說明。

兩個最常用的離散分佈：

- **白努利分佈 Bernoulli distribution**（也有人寫「伯努利」）：只有 1 和 0 兩種結果，1 的機率是 $p$、0 的機率是 $1 - p$。丟硬幣、「下一個 token 是不是『冷』」都是。
- **類別分佈 categorical distribution**：$K$ 種結果，各有一個機率 $q_1, \ldots, q_K$，全部加起來是 1。骰子是 $K = 6$；**下一個 token 也是**：EP00 的 Softmax 吐出的 151,936 個機率，就是一個 $K = 151{,}936$ 的類別分佈。白努利是 $K = 2$ 的特例。

畫成長條圖，每一根的高度是一個值的機率或比例。左圖是骰子丟六萬次的比例，右圖是 EP00「今天天氣很」的下一個 token（前 5 名，其他十幾萬個併成一格）。印出來的第一行是前 10 次的點數和每一面出現的次數，第二行是比例，第三行是 token 那 6 格的總和：

In [ ]:
rng = np.random.default_rng(0)
rolls = rng.integers(1, 7, size=60_000)            # 1 到 6（不含 7），丟六萬次
counts = np.bincount(rolls, minlength=7)[1:]       # bincount 數出 0 到 6 各出現幾次，去掉 0
print(rolls[:10], counts)
print(np.round(counts / rolls.size, 4))
tokens5 = ["冷", "熱", "晴", "暖", "乾", "其他"]
p_ep00 = np.array([0.293, 0.229, 0.065, 0.045, 0.033, 0.335])   # EP00 前 5 名，其他十幾萬個併成一格
print(p_ep00.sum())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3))
ax1.bar(range(1, 7), counts / rolls.size)
ax1.set(title="dice: 60,000 rolls", xlabel="face")
ax2.bar(range(6), p_ep00)
ax2.set(title="next token after 'today the weather is'", xlabel="0 cold 1 hot 2 sunny 3 warm 4 dry 5 other")
plt.show()

網站上的圖多畫了 1/6 的參考線和每一根的數值，橫軸改寫 token 的英文（cold 冷、hot 熱、sunny 晴、warm 暖、dry 乾、other 其他），「其他」畫成灰色：

In [ ]:
names = ["cold", "hot", "sunny", "warm", "dry", "other"]          # 冷、熱、晴、暖、乾、其他
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.bar(range(1, 7), counts / rolls.size, color="C0", width=0.7)
ax1.axhline(1 / 6, color="C3", linestyle="--", linewidth=1.2, label="1/6")
ax1.set(title="dice: 60,000 rolls", xlabel="face", ylabel="proportion", ylim=(0, 0.38), xticks=range(1, 7))
ax1.legend(frameon=False)
bars = ax2.bar(range(6), p_ep00, color="C0", width=0.7)
bars[-1].set_color("0.6")                                         # 「其他」畫成灰色
for i, v in enumerate(p_ep00):
    ax2.annotate(f"{v:.3f}", (i, v), xytext=(0, 3), textcoords="offset points", ha="center",
                 fontsize=9, color="0.25")
ax2.set(title="next token after 'today the weather is' (EP00)", ylabel="probability",
        ylim=(0, 0.38), xticks=range(6), xticklabels=names)
for ax in (ax1, ax2):
    tidy(ax)
plt.show()

骰子六根幾乎一樣高（$\frac{1}{6} \approx 0.1667$）。token 那張最右邊的「其他」反而最高：十幾萬個 token 每個都很小，併在一起卻占了三分之一。

### 取樣：依機率抽一個

EP00 的「依機率抽一個」，就是從類別分佈**取樣 sampling**，抽出來的結果叫**樣本 sample**。`rng.choice` 可以用 `p=` 指定每個結果的機率。第一行從 EP00 的分佈抽 10 次；第二行再抽一萬次（結果用編號 0–5，依序是冷、熱、晴、暖、乾、其他），印出每個編號的比例。**先猜猜看**：抽 10 次，第一個一定是機率最高的「冷」嗎？

In [ ]:
rng = np.random.default_rng(0)
print(rng.choice(tokens5, size=10, p=p_ep00))     # 從 EP00 的分佈抽 10 次
idx = rng.choice(6, size=10_000, p=p_ep00)        # 再抽一萬次，結果用編號 0–5
print(np.bincount(idx, minlength=6) / idx.size)

機率最高的「冷」不一定被抽到，3.3% 的「乾」也可能第一個出現；抽一萬次，比例就很靠近分佈本身（「冷」0.292 對 0.293）。EP00 的溫度改的是抽之前的機率本身；其他挑 token 的方法 EP28 再講。

### 期望值：用機率加權的平均

骰子平均丟出幾點？每個值乘上自己的機率，再加起來：

$$
E[X] = \sum_x x \, P(X = x) = 1 \cdot \tfrac{1}{6} + 2 \cdot \tfrac{1}{6} + \cdots + 6 \cdot \tfrac{1}{6} = 3.5
$$

白話：每個可能的值，用它的機率當權重，算加權平均。

這叫**期望值 expected value**，$E[X]$ 念「X 的期望值」。期望值不一定丟得出來（沒有 3.5 點）；丟很多次，平均會靠近它（左邊是公式，右邊是上面六萬次的平均）：

In [ ]:
faces = np.arange(1, 7)
print(faces @ np.full(6, 1 / 6), rolls.mean())

`faces @ np.full(6, 1 / 6)` 是 EP03 的內積：每一面乘上 $\frac{1}{6}$ 再加起來。本集重點的「模型預期的次數」也是期望值，EP07 的熵也是。

## 2. 工具二：聯合機率與條件機率

### 聯合機率表

一個小世界（**數字是示範用的，自己編的**）：一份文字裡，「很」「不」後面只會接「冷」「熱」「好」。從「很」或「不」出現的位置裡隨便挑一個，把它當成前一個字，記成 $A$；它後面的下一個字記成 $B$（隨機變數，只是值是字），6 種組合的機率排成一張表：

| | 冷 | 熱 | 好 | 一列加起來 $P(A)$ |
|---|---|---|---|---|
| **很** | 0.20 | 0.15 | 0.25 | 0.60 |
| **不** | 0.05 | 0.05 | 0.30 | 0.40 |
| **一欄加起來 $P(B)$** | 0.25 | 0.20 | 0.55 | 1 |

每一格是「前一個字是這一列、**而且**下一個字是這一欄」的機率，叫**聯合機率 joint probability**，寫成 $P(A, B)$，例如 $P(\text{很}, \text{冷}) = 0.20$（$P(A = \text{很}, B = \text{冷})$ 的簡寫）。高中數學用事件的寫法 $P(\text{很} \cap \text{冷})$，$\cap$ 念「交集」。6 格就是整個樣本空間，加起來是 1（印出形狀和總和）：

In [ ]:
joint = np.array([[0.20, 0.15, 0.25],     # 前一個字「很」：很冷、很熱、很好
                  [0.05, 0.05, 0.30]])    # 前一個字「不」：不冷、不熱、不好
print(joint.shape, joint.sum())

### 邊際機率：把一列加起來

「前一個字是『很』」的機率，不管下一個字是什麼：「很冷」「很熱」「很好」互斥，用規則 3 加起來，$P(\text{很}) = 0.20 + 0.15 + 0.25 = 0.60$。它寫在表的邊上，所以叫**邊際機率 marginal probability**。用 EP01 的 `axis`（左邊是 $P(\text{前一個字})$，右邊是 $P(\text{下一個字})$）：

In [ ]:
p_prev = joint.sum(axis=1)                # 每一列加起來：P(前一個字)
p_next = joint.sum(axis=0)                # 每一欄加起來：P(下一個字)
print(p_prev, p_next)

### 條件機率：只看一列，再讓它加起來是 1

已經知道前一個字是「很」，下一個字是「冷」的機率是多少？世界縮小成「很」那一列：0.20、0.15、0.25。可是這三個加起來只有 0.60，不是 1；全部除以 0.60，就是**條件機率 conditional probability**：

$$
P(B \mid A) = \frac{P(A, B)}{P(A)}, \qquad P(\text{冷} \mid \text{很}) = \frac{0.20}{0.60} \approx 0.333
$$

白話：**只看 $A$ 那一列，再除以那一列的總和**，讓它加起來是 1。

$P(B \mid A)$ 念「給定 $A$，$B$ 的機率」，高中數學念「在 $A$ 發生的條件下，$B$ 的機率」。直線 $\mid$ 右邊是已經知道的事，$P(A) > 0$ 才有定義。「除以總和」和 EP01 Softmax 的最後一步一樣。下面先印出條件機率表，再印出每一列的總和：

In [ ]:
cond = joint / joint.sum(axis=1, keepdims=True)    # (2, 3) / (2, 1)：每一列除以自己的總和
print(np.round(cond, 3))
print(cond.sum(axis=1))

`keepdims=True` 留下 `(2, 1)`，每一列才會除以**自己的**總和（EP01 的廣播；忘了會怎樣，見弄壞它 1）。每一列都是一個分佈，叫**條件分佈 conditional distribution**。同樣是「好」，前一個字是「很」時 0.417，是「不」時 0.75。

把兩張表畫出來（圖上 very、not 是「很」「不」，cold、hot、good 是「冷」「熱」「好」；顏色越深，機率越大）：

In [ ]:
def draw_table(ax, values, title, row_sums, col_sums=None):
    """把一張 2 × 3 的機率表畫成格子，右邊寫每一列的總和；有給 col_sums 的話，下面寫每一欄的總和。"""
    rows, cols = ["A = very", "A = not"], ["B = cold", "B = hot", "B = good"]   # 很、不；冷、熱、好
    ax.imshow(values, cmap="Blues", vmin=0, vmax=1.0, aspect="auto")
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            ax.text(j, i, f"{v:.3g}", ha="center", va="center", fontsize=12,
                    color="white" if v > 0.55 else "0.1")
        ax.text(values.shape[1] - 0.35, i, f"row sum {row_sums[i]:g}", ha="left", va="center",
                fontsize=10, color="0.3")
    if col_sums is not None:
        for j, s in enumerate(col_sums):
            ax.text(j, values.shape[0] - 0.35, f"{s:g}", ha="center", va="top", fontsize=10, color="0.3")
        ax.text(-0.5, values.shape[0] - 0.35, "column sum", ha="right", va="top", fontsize=10, color="0.3")
    ax.set_xticks(range(3), cols)
    ax.set_yticks(range(2), rows)
    ax.xaxis.tick_top()
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, pad=28)
    for x in [0.5, 1.5]:
        ax.axvline(x, color="white", linewidth=2)
    ax.axhline(0.5, color="white", linewidth=2)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.3))
draw_table(ax1, joint, "joint P(A, B) (demo numbers): all 6 cells sum to 1", joint.sum(axis=1), joint.sum(axis=0))
draw_table(ax2, np.round(cond, 3), "conditional P(B | A): divide each row by its sum", cond.sum(axis=1))
fig.subplots_adjust(wspace=0.55)
plt.show()

左邊 6 格加起來是 1；右邊每一列加起來是 1，例如 $P(\text{冷} \mid \text{很}) = 0.333$、$P(\text{好} \mid \text{不}) = 0.75$。

### 獨立：知道了也沒用

如果知道 $A$ 不會改變 $B$ 的機率，$P(B \mid A) = P(B)$，就說 $A$、$B$ **獨立 independent**。代進條件機率的定義（$P(A) > 0$ 時），等價的寫法是

$$
P(A, B) = P(A)\,P(B)
$$

白話：聯合機率＝邊際機率相乘，表裡**每一格**都要成立。

這個寫法 $P(A) = 0$ 時也能用，所以正式的定義用它。丟兩次硬幣是獨立的：第一次是正面，不會改變第二次。

> **注意**：獨立不是互斥。「前一個字是『很』」和「前一個字是『不』」互斥，同時發生的機率是 0，不是 $0.6 \times 0.4$：知道一個發生，另一個就不會發生，反而不獨立。

這張表獨立嗎？`np.outer(p_prev, p_next)` 的第 $i$ 列第 $j$ 欄是 `p_prev[i] × p_next[j]`：如果獨立，表應該長這樣。最後一行比較它和真正的表：

In [ ]:
print(np.outer(p_prev, p_next))           # 如果獨立，表應該長這樣
print(np.allclose(np.outer(p_prev, p_next), joint))

不獨立：$P(\text{很}, \text{冷})$ 是 0.20，不是 0.15；$P(\text{好} \mid \text{不}) = 0.75$，比 $P(\text{好}) = 0.55$ 大很多。**語言幾乎從來不是獨立的**：「今天天氣很」後面是「冷」的機率（29.3%），和隨便一個位置剛好是「冷」的機率差得很遠。所以語言模型要算的是條件機率（見弄壞它 2）。

（網站上有互動元件：點一列看條件分佈，點一欄看反過來的 $P(A \mid B)$（已知下一個字，除以那一欄的總和），也可以切換成「如果獨立」的表。）

## 3. 工具三：機率的鏈鎖律

### 乘法規則

把條件機率的定義兩邊乘上 $P(A)$：

$$
P(A, B) = P(A)\,P(B \mid A)
$$

白話：「$A$ 而且 $B$」的機率＝先發生 $A$ 的機率 × 在 $A$ 發生的條件下、再發生 $B$ 的機率。

這叫**乘法規則 multiplication rule**（和 EP05 的乘法法則只差一個字：那是導數，這是機率；不少英文書也把這條叫 product rule，和下面的鏈鎖律一樣撞名）。用表驗算，$P(\text{很}, \text{冷}) = 0.6 \times \frac{0.20}{0.60} = 0.2$（先印出 $P(A)$ 乘上 $P(B \mid A)$，再和聯合機率表比較）：

In [ ]:
print(p_prev[:, None] * cond)                      # P(A) 乘上 P(B | A)：(2, 1) * (2, 3)
print(np.allclose(p_prev[:, None] * cond, joint))

`p_prev[:, None]` 就是 `reshape(2, 1)`。順序也可以反過來拆，$P(A, B) = P(B)\,P(A \mid B)$；兩種拆法放在一起就是貝氏定理 Bayes theorem，這個系列暫時用不到。

### 推廣到 n 個

三件事 $x_1, x_2, x_3$（例如一句話的前三個 token）：把「$x_1$ 而且 $x_2$」整個當成 $A$、$x_3$ 當成 $B$，用一次乘法規則；再把 $P(x_1, x_2)$ 拆一次：

$$
P(x_1, x_2, x_3) = P(x_1, x_2)\,P(x_3 \mid x_1, x_2) = P(x_1)\,P(x_2 \mid x_1)\,P(x_3 \mid x_1, x_2)
$$

白話：第一件的機率，乘上後面每一件「給定前面幾件」的條件機率。

$n$ 件事一路拆下去：

$$
P(x_1, \ldots, x_n) = P(x_1)\,P(x_2 \mid x_1) \cdots P(x_n \mid x_1, \ldots, x_{n-1}) = \prod_{t=1}^{n} P(x_t \mid x_{<t})
$$

白話：每一項都是「給定前面全部，下一個的條件機率」，全部乘起來。

$x_{<t}$（念「x 下標小於 t」：編號在 $t$ 之前，不是值小於 $t$）是 $x_1, \ldots, x_{t-1}$；$t = 1$ 時前面什麼都沒有，那一項就是 $P(x_1)$。這叫**機率的鏈鎖律 chain rule of probability**。它**不需要獨立之類的假設**，只用了條件機率的定義（前面的機率大於 0，每一項才有定義），任何 $n$ 件事都成立。

> **注意**：英文也叫 chain rule，但和 EP05 的鏈鎖律是兩件事，只是名字一樣：EP05 是導數一路相乘，這裡是機率一路相乘。

### 兌現 EP02：一句話的機率

EP02 的「今天／天／氣／很／冷」，$p_1, \ldots, p_5$ 其實是這 5 個條件機率（前 4 個是示範數字）：

| $t$ | token $x_t$ | $p_t$ | 數字 |
|---|---|---|---|
| 1 | 今天 | $P(\text{今天})$：一句話用「今天」開頭 | 0.01（示範） |
| 2 | 天 | $P(\text{天} \mid \text{今天})$ | 0.2（示範） |
| 3 | 氣 | $P(\text{氣} \mid \text{今天天})$ | 0.9（示範） |
| 4 | 很 | $P(\text{很} \mid \text{今天天氣})$ | 0.3（示範） |
| 5 | 冷 | $P(\text{冷} \mid \text{今天天氣很})$ | 0.293（EP00 實測） |

$$
P(\text{今天天氣很冷}) = P(\text{今天})\,P(\text{天} \mid \text{今天}) \cdots P(\text{冷} \mid \text{今天天氣很}) = \prod_{t=1}^{5} P(x_t \mid x_{<t})
$$

白話：一句話的機率＝每一步「給定前面所有 token，下一個是正確 token」的條件機率連乘。

所以 EP02「每一步猜中下一個 token 的機率」連乘，**就是機率的鏈鎖律**；模型每一步看的，正好就是前面所有的 token（EP00 的自回歸）。第一個 token 沒有前文，實作上常在最前面放一個代表「開頭」的特殊 token special token（詞彙表裡專門留的記號）當條件。嚴格說，這 5 項連乘是「開頭 5 個 token 剛好是這 5 個」的機率；要算「整句話到這裡結束」，還要再乘上下一個是「結束」特殊 token 的條件機率。

第一行是連乘和 log 機率的總和，第二行是 log 機率一步一步加起來（`np.cumsum`）：

In [ ]:
p_cond = np.array([0.01, 0.2, 0.9, 0.3, 0.293])    # 只有最後一個是 EP00 的實測，其他是示範
print(np.prod(p_cond), np.sum(np.log(p_cond)))
print(np.round(np.cumsum(np.log(p_cond)), 3))      # log 機率一步一步加起來

實際計算時都用 EP02 的 **log 機率**：連乘變加總，再長的句子也不會下溢。最後一步的 $\ln 0.293 = -1.228$，正是 EP02 算過的那個數。畫出來（左圖藍色的前 4 根是示範數字，紅色的最後一根是 EP00 的實測；圖上 today、wea-、-ther、very、cold 是「今天」「天」「氣」「很」「冷」）：

In [ ]:
labels = ["t=1\n'today'", "t=2\n'wea-'", "t=3\n'-ther'", "t=4\n'very'", "t=5\n'cold'"]   # 今天、天、氣、很、冷
cum = np.cumsum(np.log(p_cond))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.8))
ax1.bar(range(5), p_cond, color=["C0"] * 4 + ["C3"], width=0.65)
for i, v in enumerate(p_cond):
    ax1.annotate(f"{v:g}", (i, v), xytext=(0, 3), textcoords="offset points", ha="center",
                 fontsize=10, color="0.25")
ax1.bar([0], [0], color="C0", label="demo numbers")          # 高度 0 的長條：只為了圖例
ax1.bar([0], [0], color="C3", label="measured in EP00")
ax1.set(xticks=range(5), ylim=(0, 1.13), ylabel="P(x_t | tokens before it)",
        title="conditional probability at each step")
ax1.set_xticklabels(labels)
ax1.legend(frameon=False, loc="upper center", ncol=2, fontsize=9)
ax1.set_yticks(np.arange(0, 1.01, 0.2))
ax2.plot(range(5), cum, "o-", color="C2", linewidth=2, markersize=7)
for i, v in enumerate(cum):
    ax2.annotate(f"{v:.3f}", (i, v), xytext=(8, 4), textcoords="offset points", fontsize=10,
                 color="0.25")
ax2.set(xticks=range(5), xlim=(-0.4, 4.7), ylim=(-10, 0), ylabel="sum of log P so far",
        title=f"running sum of log probabilities (product = {np.prod(p_cond):.6f})")
ax2.set_xticklabels(labels)
for ax in (ax1, ax2):
    tidy(ax)
plt.show()

右圖的 log 機率一步一步往下加，從 −4.605 降到 −8.752；連乘是 0.000158。

（網站上有互動元件：把句子加長，看連乘下溢成 0、log 加總不會；也可以切換成「當成獨立」，把詞序對調看看。先猜猜會怎樣，弄壞它 2 會解釋。）

### 語言模型

**語言模型 language model**：給前面的 token $x_{<t}$，輸出下一個 token 的條件分佈 $P(\cdot \mid x_{<t})$（$\cdot$ 代表「詞彙表裡的每一個 token」）。EP00 的模型每跑一次，Softmax 吐出的 151,936 個機率就是這個條件分佈。有了它可以：

- **算一句話的機率**：每一步取出正確 token 的條件機率，用鏈鎖律連乘（實際上是 log 加總）。
- **生成**：從條件分佈取樣一個 token，接到後面，再算下一步⋯⋯就是 EP00 的自回歸。

模型的參數（約 5 億個數字）決定每一步的條件分佈：參數換一組，同一句話的機率就變了。哪一組參數比較「好」？

## 4. 工具四：最大概似估計

### 同一份資料，不同參數

一枚硬幣不知道公不公平，丟 10 次：正正反正正正反正正反，7 次正面。假設它正面的機率是 $p$，$p$ 就是這枚硬幣的**參數**。每次丟都是獨立的，所以這份資料的機率是 10 個數連乘：

$$
P(\text{資料}; p) = p \cdot p \cdot (1 - p) \cdot p \cdot p \cdot p \cdot (1 - p) \cdot p \cdot p \cdot (1 - p) = p^7 (1 - p)^3
$$

白話：正面乘 $p$、反面乘 $1 - p$，10 次全部乘起來。

分號後面寫參數：「參數是 $p$ 時」。這也是鏈鎖律：因為獨立，每一項 $P(x_t \mid x_{<t})$ 都變成 $P(x_t)$，不用看前面（語言就不行）。`np.where(條件, a, b)`：條件成立取 `a`、否則取 `b`。$p$ 換不同的值（每一行是 $p$ 和這份資料的機率）。**先猜猜看**：哪一個 $p$ 最大？

In [ ]:
flips = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 0])   # 正正反正正正反正正反：7 次正面

def likelihood(p):
    return np.prod(np.where(flips == 1, p, 1 - p))  # 正面乘 p、反面乘 1 − p

for p in [0.3, 0.5, 0.7, 0.9]:
    print(p, likelihood(p))

$p = 0.7$ 時，這份資料的機率最大，是 $p = 0.5$ 的 2.3 倍。

### 概似函數

同一條式子 $p^7 (1 - p)^3$，有兩種看法：

- **參數固定、看資料**：$p = 0.5$ 時，丟出「正正反正正正反正正反」這一串的機率是多少？這是**機率**。
- **資料固定、看參數**：資料已經是這一串，$p$ 換成多少，這份資料的機率最大？這時把它看成 $p$ 的函數，叫**概似函數 likelihood function**：

$$
L(p) = p^7 (1 - p)^3
$$

白話：資料固定，參數是 $p$ 時，這份資料的機率。

這是「這一串、照這個順序」的機率；不管順序的話要再乘上 7 正 3 反的排法數 120（高中的二項分佈 binomial distribution）；120 和 $p$ 無關，整條曲線只是放大 120 倍，最大值還在同一個 $p$。「概似」是「看起來有多像」：$L(0.7) > L(0.5)$，$p = 0.7$ 比較能解釋這份資料。

> **注意**：$L(0.7)$ **不是**「$p$ 是 0.7 的機率」，而是 $p = 0.7$ 時這份資料的機率。$p$ 固定時，丟 10 次全部 1024 種結果的機率加起來是 1；資料固定、$p$ 換來換去，$L(p)$ 沒有「加起來是 1」這回事。

**最大概似估計 MLE maximum likelihood estimation**：選讓概似最大的參數，

$$
\hat{p} = \operatorname*{argmax}_{p} L(p)
$$

白話：$\hat{p}$ 是讓 $L(p)$ 最大的那個 $p$。

$\hat{p}$ 念「p hat」，戴帽子代表「從資料估計出來的」；argmax 回傳的是位置，不是最大值本身（和 EP01 的 `argmax` 一樣）。先用**格點 grid** 找：0 到 1 每隔 0.01 取一個點，每一點都算 $L(p)$，挑最大的（印出最大的位置和最大值）：

In [ ]:
ps = np.arange(0, 101) / 100                      # 0、0.01、⋯、1
L = np.array([likelihood(p) for p in ps])
print(ps[np.argmax(L)], L.max())

### 取 log：山頂的位置不變

$L(p)$ 是連乘，取 log 變加總（EP02 的對數律；沒寫底數的 $\log$ 就是 $\ln$），叫 **log 概似 log-likelihood**：

$$
\log L(p) = 7 \ln p + 3 \ln(1 - p)
$$

白話：7 個 $\ln p$ 加上 3 個 $\ln(1 - p)$。

$\ln$ 是遞增函數（EP02）：$L$ 越大，$\log L$ 越大，所以**山頂在同一個 $p$**。而且資料一多，$L$ 會下溢成 0，$\log L$ 不會（弄壞它 3）。

山頂是平的（EP04：導數是 0）。$(\ln p)' = \frac{1}{p}$；$\ln(1 - p)$ 用 EP05 的鏈鎖律，內層 $1 - p$ 的導數是 $-1$，所以是 $-\frac{1}{1 - p}$：

$$
\frac{d}{dp}\log L = \frac{7}{p} - \frac{3}{1 - p} = 0 \;\Rightarrow\; 7(1 - p) = 3p \;\Rightarrow\; \hat{p} = \frac{7}{10} = 0.7
$$

白話：令導數為 0，解出 $\hat{p} = 0.7$（$\Rightarrow$ 念「所以」）。

用中央差分驗算：前四行是 $p$、數值導數、公式 $\frac{7}{p} - \frac{3}{1 - p}$；最後一行比較 `log_lik(0.7)` 和直接對 $L(0.7)$ 取 log：

In [ ]:
def log_lik(p):
    return 7 * np.log(p) + 3 * np.log(1 - p)

for p in [0.5, 0.6, 0.7, 0.8]:
    print(p, nd(log_lik, p), 7 / p - 3 / (1 - p))
print(log_lik(0.7), np.log(likelihood(0.7)))

左邊導數正（上坡）、右邊負（下坡），0.7 兩欄都幾乎是 0（$10^{-9}$ 是中央差分的誤差，$10^{-15}$ 是捨入誤差）。$p$ 變大時，$\frac{7}{p}$ 一路變小、$\frac{3}{1 - p}$ 一路變大，導數只在 0.7 穿過 0，所以 0.7 是整條曲線的山頂（EP04：導數是 0 只是「可能」）。

一般來說，丟 $n$ 次、$k$ 次正面，$0 < k < n$ 時同樣解出 $\hat{p} = \frac{k}{n}$；全部正面時，導數 $\frac{n}{p}$ 永遠是正的，一路上坡到端點 $p = 1$，答案還是 $\frac{k}{n}$（全部反面對稱，$\hat{p} = 0$；弄壞它 4）。**白努利分佈的最大概似估計，就是正面的比例**：工具一的「用比例猜機率」，原來就是「讓資料的機率最大」的答案。

畫成圖：左邊 $L(p) = p^7(1 - p)^3$ 是一座山，山頂在 $\hat{p} = 0.7$、高約 0.00222；右邊 $\log L(p)$ 也在 0.7 最高（約 −6.11），往兩端往下掉。這張圖由第 6 節動手做的 `draw_likelihood`、`draw_log_likelihood` 來畫，先自己寫寫看（參考解答在最後）。

（網站上有互動元件：拖動 $p$ 看 $L(p)$ 與 $\log L(p)$ 兩條曲線和目前的值，也可以調丟的次數和正面的次數。）

### 類別分佈：也是比例

$K$ 種結果時，參數是 $K$ 個機率 $q_1, \ldots, q_K$（加起來是 1）。第 $j$ 種在資料裡出現 $c_j$ 次（和硬幣一樣，每個位置當成獨立、不看前文），每出現一次就乘一個 $q_j$，所以

$$
\log L = \sum_{j=1}^{K} c_j \ln q_j
$$

白話：每種結果的次數乘上它的 log 機率，全部加起來。

答案也是比例 $\hat{q}_j = \frac{c_j}{n}$，$n = c_1 + \cdots + c_K$ 是資料的總數（證明要處理「加起來是 1」的限制，本集重點會用數值驗證）。例子：詞彙表只有「冷」「熱」「好」3 個 token，資料是一小串 token（示範）。印出每個 token 的次數、比例，以及比例那一點的 log 概似：

In [ ]:
data = np.array([0, 1, 0, 2, 0, 2, 2, 0, 1, 0])  # 冷 熱 冷 好 冷 好 好 冷 熱 冷（0 冷、1 熱、2 好）
c = np.bincount(data, minlength=3)               # 每個 token 出現幾次
q_hat = c / c.sum()
print(c, q_hat, c @ np.log(q_hat))

`c @ np.log(q_hat)` 就是上面的 $\sum_j c_j \ln q_j$（內積）。換成條件分佈也一樣：「前一個字是『很』時，下一個字的分佈」的最大概似估計，就是「很」後面接每個字的比例。**EP17 的第一個語言模型就是這樣做的**：拿真的文字，算出每個字後面接每個字的次數，再換成比例。

## 5. 本集重點：訓練語言模型＝最大概似估計

### 四個工具接起來

- 模型的參數記成 $\theta$（希臘字母，念 theta，代表全部約 5 億個數字；EP03 用它代表角度，這裡是另一個意思），它決定每一步的條件分佈 $P_\theta(\cdot \mid x_{<t})$：右下角的 $\theta$ 和工具四的分號一樣，是「參數是 $\theta$ 時」。
- 一句話的機率是條件機率的連乘（工具三），取 log 變加總。句子和句子之間，訓練時當成彼此獨立（一種簡化），log 再加總。
- 最大概似估計：選讓全部訓練資料的 log 機率總和最大的參數（工具四）：

$$
\hat{\theta} = \operatorname*{argmax}_{\theta} \sum_{\text{每一句}} \sum_{t} \log P_\theta(x_t \mid x_{<t})
$$

白話：**修改參數，讓模型給訓練資料裡「真正的下一個 token」的 log 機率，全部位置加起來越大越好。** $\hat{\theta}$ 和工具四的 $\hat{p}$ 一樣戴帽子：從資料估計出來的參數。

這就是 EP05 預告的訓練目標。把這個總和取負號、再除以 token 數，就是 LLM 訓練時看到的損失，EP07 會說明它為什麼叫交叉熵。

硬幣可以令導數為 0 直接解；5 億個參數沒辦法。做法是 EP04、EP05 的：用反向傳播算出 log 概似對每個參數的梯度，往梯度的方向把參數改一點點，重複很多次。一步走多遠、走幾步，是 EP08 梯度下降的事。

### 玩具模型：3 個分數、一個 Softmax

最小的「語言模型」：詞彙表是上面的 3 個 token，**不看前文**，參數是 3 個分數 $z$，機率是 Softmax($z$)，資料是上面那 10 個 token。log 概似用 EP02 的 log-softmax 算，梯度用 EP04 的數值梯度，每一步往梯度的方向走 0.1 倍、走 50 步。這裡只示範一次；0.1 怎麼選、什麼時候停，是 EP08 學習率的事。

`log_softmax(z)[data]`：`data` 是整數陣列，一次取出 10 個位置的 log 機率。印出來的第一行是起點的梯度；接著每一行是步數、三個機率、log 概似；最後一行是「分數取 log(比例)」那一點的梯度（那一點的 Softmax 剛好是比例）。圖是每一步的 log 概似，紅色虛線是工具四的最大值：

In [ ]:
def log_lik_z(z):
    return np.sum(log_softmax(z)[data])            # data 是整數陣列：一次取出 10 個位置的 log 機率

print(numerical_gradient(log_lik_z, np.zeros(3)))
z, history = np.zeros(3), []                       # 一開始三個分數一樣：機率各 1/3
for step in range(51):
    history.append(log_lik_z(z))                   # 實驗紀錄：每一步的 log 概似
    if step in [0, 1, 2, 5, 10, 20, 50]:
        print(step, np.round(softmax(z), 3), round(log_lik_z(z), 4))
    z = z + 0.1 * numerical_gradient(log_lik_z, z)  # 往 log 概似上升最快的方向走一小步
print(numerical_gradient(log_lik_z, np.log(q_hat)))  # 分數取 log(比例)，Softmax 剛好是比例
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(history)
ax.axhline(c @ np.log(q_hat), color="red", ls="--")  # 工具四的最大值
ax.set(xlabel="step", ylabel="log-likelihood", title="toy model climbs to the MLE")
plt.show()

- 第一行：一開始的梯度 $(1.67, -1.33, -0.33)$ 剛好是「實際出現次數 − 模型預期的次數」：$5 - \frac{10}{3}$、$2 - \frac{10}{3}$、$3 - \frac{10}{3}$（每個位置是「冷」的機率都是 $\frac{1}{3}$，10 個位置預期 $\frac{10}{3}$ 次，就是期望值）。猜得太少的，分數就調高；為什麼剛好是這樣，EP07 推 Softmax 的導數時會看到。
- log 概似從 $10 \ln \frac{1}{3} = -10.9861$ 一路變大，停在工具四的最大值 $-10.2965$，機率停在資料的比例。最後一行驗證山頂：在「比例」那一點梯度是 0（$10^{-11}$ 是中央差分的誤差）。**用梯度一步一步爬，爬到的就是最大概似估計**（這個玩具的 log 概似只有一座山）。
- 「冷」為什麼沒被推到 1？5 個位置要「冷」、2 個要「熱」、3 個要「好」，三個機率加起來只能是 1，最好的妥協就是比例。真正的語言模型也一樣：同一段前文在訓練資料裡接過很多種字，最大概似要的是照比例分（EP00 的模型給「今天天氣很」後面的「冷」也只有 29.3%；實際的模型只會大致接近比例，見下面）。
- 上面的圖就是程式畫的實驗紀錄：每一步的 log 概似。

網站上的圖右邊另外畫了每一步的 `softmax(z)`。同樣的起點、同樣的 50 步再走一次，這次記下每一步的機率（另外取名字 `z_run`，不改到上面走完的 `z`，和 PyTorch 對照時還要用）：

In [ ]:
z_run, probs = np.zeros(3), []                     # 同樣的起點、同樣的 50 步
for step in range(51):
    probs.append(softmax(z_run))                   # 記下每一步的冷、熱、好
    z_run = z_run + 0.1 * numerical_gradient(log_lik_z, z_run)
probs = np.array(probs)                            # (51, 3)：第 t 列是第 t 步的三個機率
best = c @ np.log(q_hat)                           # 工具四的最大值
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 3.6))
ax1.plot(history, color="C0", linewidth=2, label="log-likelihood")
ax1.axhline(best, color="C3", linestyle="--", linewidth=1.2, label=f"maximum {best:.3f}")
ax1.annotate(f"start {history[0]:.3f}", (0, history[0]), xytext=(10, 0), textcoords="offset points",
             va="center", fontsize=10, color="0.25")
ax1.set(xlabel="step", ylabel="log-likelihood", title="toy model climbs to the MLE", ylim=(-11.1, -10.15))
ax1.legend(frameon=False, loc="lower right")
names3 = ["cold (5 of 10)", "hot (2 of 10)", "good (3 of 10)"]   # 冷、熱、好
for k in range(3):
    ax2.plot(probs[:, k], color=f"C{k}", linewidth=2, label=names3[k])
    ax2.hlines(q_hat[k], 0, 50, color=f"C{k}", linestyle=":", linewidth=1)   # 點線：資料的比例
    ax2.annotate(f"{q_hat[k]:g}", (50, probs[-1, k]), xytext=(6, 0), textcoords="offset points",
                 va="center", fontsize=10, color="0.25")
ax2.set(xlabel="step", ylabel="probability", title="probabilities move to the data proportions",
        ylim=(0, 0.6), xlim=(0, 55))
ax2.legend(frameon=False, loc="upper right", ncol=3, fontsize=9)
for ax in (ax1, ax2):
    tidy(ax)
plt.show()

左圖從 −10.986 開始，10 步左右就幾乎碰到紅色虛線的最大值 −10.297；右圖冷、熱、好三個機率從各 1/3 走到 0.5、0.2、0.3，就是資料裡 5、2、3 次的比例。

### 和 LLM 的連結

- 真正的語言模型一樣，只是分數是模型看了前文算出來的（EP00 的 logits），參數有 5 億個，梯度用反向傳播算（EP05）。
- 真正的地形複雜得多，訓練只爬到夠高就停，也不會去追訓練資料上的最高點（太貼合訓練資料反而不好，是 EP16 的過擬合）。所以精確的說法是：**訓練的目標**是最大概似。
- **預訓練的目標就是最大概似**（EP38 的監督式微調也是；EP41 起的偏好最佳化換了目標）。EP00 說預訓練模型學的是「網路上的文字接下來通常怎麼寫」，就是這個意思：它的前 10 名裡出現簡體的「干燥」，因為訓練資料裡就有人這樣寫。

## 6. 動手做：從零寫出 `prob.py` 的 5 個函式

在下一格寫完這 5 個函式（網站上是寫在 `llm-labs/ep06/prob.py`），再執行下一格的檢查。

要求與提示：

1. **只用 NumPy 和基本的 Python**。
2. **`empirical_distribution`**：樣本裡每個值出現的比例排成的分佈，叫**經驗分佈 empirical distribution**。`np.bincount(samples, minlength=k)` 算出每個值出現幾次，再除以樣本數（不是除以 `k`）。整數檢查用 `np.issubdtype(samples.dtype, np.integer)`。
3. **`conditional`**：工具二的一行，`keepdims=True` 不能少。先用 `np.asarray(joint, dtype=float)` 轉成小數，次數表（整數）也能用；它遇到小數陣列不會複製，所以不要用 `/=`（會改到呼叫端的表）。加總是 0 的列直接除，會得到 `nan` 和 `RuntimeWarning`，要先檢查。
4. **`chain_log_prob`**：可以用最上面定義的 `sequence_logprob`（EP02 的 Σ log；網站上是 `from ep02.mathlang import sequence_logprob`），再加上範圍檢查和「有 0 就回傳 `-np.inf`」（先檢查，就不會呼叫到 `np.log(0)` 而跳出警告）。檢查會傳 Python list，先 `np.asarray(cond_probs, dtype=float)`。
5. **`log_likelihood`**：直接照公式寫，`p = 1` 時 `np.log(1 - p)` 是 `-inf`（還會跳出警告），再乘上反面數 0 得到 `nan`，不是 0（`p = 0` 同理）。寫一個小函式 `term(count, prob)`：`count` 是 0 就回傳 `0.0`、`prob` 是 0 就回傳 `-np.inf`，其他照公式。`flips` 可能是 list，先 `np.asarray(flips)`。

In [ ]:
import numpy as np


def empirical_distribution(samples, k):
    """samples：一串整數，每個都在 0 到 k − 1（k 是類別數，工具四的 K）。回傳長度 k 的陣列，第 i 格是 i 出現的比例
    （沒出現就是 0），加起來是 1。samples 是空的、有值不在 0 到 k − 1、或不是整數，要 raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 empirical_distribution")


def conditional(joint):
    """joint：二維聯合機率表，第 i 列第 j 欄是 P(A = i, B = j)。回傳同形狀的條件機率表，
    第 i 列是 P(B | A = i)：每一列除以那一列的加總 P(A = i)，所以每一列加起來是 1。
    某一列加總是 0（P(A = i) = 0，條件機率沒有定義）要 raise ValueError。不能改到呼叫端的 joint。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 conditional")


def chain_log_prob(cond_probs):
    """一串條件機率 P(x₁)、P(x₂ | x₁)、⋯、P(xₙ | x₁, ⋯, xₙ₋₁) → log P(x₁, ⋯, xₙ) = Σ log。
    不先連乘（長序列不能下溢）。每個機率都要在 0 到 1 之間，否則 raise ValueError；
    有 0 時回傳 -inf，而且不能跳出 RuntimeWarning。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 chain_log_prob")


def bernoulli_mle(flips):
    """flips：0/1 陣列（1 = 正面）。回傳正面的比例，也就是白努利分佈的最大概似估計 p̂。
    空的、或有 0 和 1 以外的值，要 raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 bernoulli_mle")


def log_likelihood(p, flips):
    """白努利分佈的 log 概似：(正面數) ln p + (反面數) ln(1 − p)。
    p 可以是 0 或 1，0 × log 0 當成 0：p = 1 而且全部正面是 0.0、p = 1 卻有反面是 -inf（p = 0 對稱）；
    不能跳出 RuntimeWarning。p 不在 [0, 1] 要 raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 log_likelihood")

下一格的 8 個檢查和網站上的 `tests/test_ep06_prob.py` 一樣，依序檢查：經驗分佈的比例（沒出現的值也要有一格），以及三種要報錯的輸入；十萬個樣本的經驗分佈很靠近真正的分佈；條件機率表每一列加起來是 1、乘回 $P(A)$ 就是聯合機率表、不能改到呼叫端的表，整數的次數表也要能用；某一列加總是 0 要報錯；一千個 0.1 的 log 機率不會下溢、有 0 時回傳 $-\infty$ 而且沒有警告、機率超出 0 到 1 要報錯；白努利的最大概似估計是 0.7，而且在格點裡 log 概似最大；log 概似在 $p = 0$、$p = 1$ 的邊界；最後和 PyTorch 比對（沒裝 PyTorch 就跳過）。幾個新寫法：

- `with raises(ValueError):` 是「底下這段程式應該要報 `ValueError`」，沒報錯，檢查就失敗。（網站上的測試用 pytest 的 `pytest.raises`，這裡寫了一個用法一樣的小工具，notebook 就不用裝 pytest。）
- `warnings.simplefilter("error")` 把警告變成錯誤；放在 `with warnings.catch_warnings():` 底下，只影響那一段。所以「不能跳出 `RuntimeWarning`」也檢查得到。
- 第三個檢查多用了一張 3 × 3 的表：忘了 `keepdims` 時，方陣不會報錯，只會算錯（弄壞它 1）。
- 最後一個檢查兩邊都照公式算，只差捨入誤差，所以用很嚴的 `rel_tol=1e-12`。

沒通過的檢查會印出出錯的那一行。改了上一格之後，要重新執行上一格，再執行這一格。

In [ ]:
import contextlib
import traceback
import warnings


def check_prob():
    """和網站上 tests/test_ep06_prob.py 一樣的 8 個測試。"""

    class Skipped(Exception):
        pass

    @contextlib.contextmanager
    def raises(exc_type):
        """和 pytest.raises 一樣：底下這段程式應該要報 exc_type，沒報錯就算失敗。"""
        try:
            yield
        except exc_type:
            pass
        else:
            raise AssertionError(f"應該要報 {exc_type.__name__}，卻沒有報錯")

    JOINT = np.array([[0.20, 0.15, 0.25],      # 工具二的示範表：前一個字「很」「不」× 下一個字「冷」「熱」「好」
                      [0.05, 0.05, 0.30]])
    FLIPS = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 0])   # 工具四的 10 次：7 次正面

    def empirical_distribution_counts_and_sums_to_one():
        p = empirical_distribution(np.array([0, 2, 2, 1, 2]), k=4)
        assert p.shape == (4,)
        assert np.allclose(p, [0.2, 0.2, 0.6, 0.0])               # 沒出現的 3 也要有一格
        assert math.isclose(p.sum(), 1.0)
        for bad in [np.array([0, 1, 4]), np.array([], dtype=int), np.array([0.0, 1.0])]:
            with raises(ValueError):                              # 4 不在 0 到 3、空的、不是整數
                empirical_distribution(bad, k=4)

    def empirical_distribution_approaches_true_distribution():
        true_p = np.array([0.5, 0.2, 0.3])
        rng = np.random.default_rng(0)                            # 固定 seed：每次跑都一樣
        samples = rng.choice(3, size=100_000, p=true_p)
        assert np.allclose(empirical_distribution(samples, k=3), true_p, rtol=0, atol=0.01)

    def conditional_rows_sum_to_one_and_rebuild_joint():
        rng = np.random.default_rng(1)
        square = rng.random((3, 3))
        square = square / square.sum()                            # 3 × 3：忘了 keepdims 也不會報錯，只會算錯
        for joint in [JOINT, square]:
            before = joint.copy()
            cond = conditional(joint)
            assert np.array_equal(joint, before)                  # 不能改到呼叫端的表
            assert cond.shape == joint.shape
            assert np.allclose(cond.sum(axis=1), 1.0)             # 每一列是一個分佈
            p_a = joint.sum(axis=1, keepdims=True)                # 邊際機率 P(A)
            assert np.allclose(p_a * cond, joint)                 # 乘法規則：P(A, B) = P(A) P(B | A)
        assert np.allclose(conditional(JOINT)[1], [0.125, 0.125, 0.75])    # P(下一個字 | 不)
        assert np.allclose(conditional(np.array([[4, 3, 5], [1, 1, 6]])), conditional(JOINT))   # 整數的次數表（JOINT × 20）也行

    def conditional_zero_row_raises():
        with raises(ValueError):
            conditional(np.array([[0.5, 0.5], [0.0, 0.0]]))       # P(A = 1) = 0：沒有定義

    def chain_log_prob_no_underflow():
        assert math.isclose(chain_log_prob([0.5, 0.2]), math.log(0.1))
        lp = chain_log_prob(np.full(1000, 0.1))
        assert np.prod(np.full(1000, 0.1)) == 0.0                 # 直接連乘：下溢成 0
        assert math.isclose(lp, 1000 * math.log(0.1))             # log 加總：−2302.58⋯
        with warnings.catch_warnings():
            warnings.simplefilter("error")                        # 有 warning 就算失敗
            assert chain_log_prob([0.5, 0.0, 0.3]) == -np.inf     # 有一步不可能 → 整句不可能
        for bad in [[0.5, 1.2], [0.5, -0.1]]:                     # 機率不能超過 1，也不能是負的
            with raises(ValueError):
                chain_log_prob(bad)

    def bernoulli_mle_maximizes_log_likelihood():
        p_hat = bernoulli_mle(FLIPS)
        assert math.isclose(p_hat, 0.7)
        for bad in [np.array([]), np.array([1, 0, 2])]:           # 空的、有 0 和 1 以外的值
            with raises(ValueError):
                bernoulli_mle(bad)
        assert math.isclose(log_likelihood(0.7, FLIPS), 7 * math.log(0.7) + 3 * math.log(0.3))
        grid = np.arange(1, 100) / 100                            # 0.01、0.02、⋯、0.99
        lls = [log_likelihood(p, FLIPS) for p in grid]
        assert math.isclose(grid[np.argmax(lls)], p_hat)          # 格點裡最大的就在 p̂
        assert all(log_likelihood(p_hat, FLIPS) >= ll for ll in lls)

    def log_likelihood_edge_cases():
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            assert log_likelihood(1.0, [1, 1, 1]) == 0.0          # 0 × log 0 當成 0：機率 1，log 是 0
            assert log_likelihood(1.0, [1, 1, 1, 0]) == -np.inf   # p = 1 卻出現反面：不可能
            assert log_likelihood(0.0, [0, 0]) == 0.0
        for bad_p in [1.5, -0.1]:
            with raises(ValueError):
                log_likelihood(bad_p, FLIPS)

    def matches_pytorch():
        try:
            import torch
        except ImportError:
            raise Skipped("沒有安裝 PyTorch")
        flips = torch.tensor(FLIPS, dtype=torch.float64)
        bern = torch.distributions.Bernoulli(probs=torch.tensor(0.7, dtype=torch.float64))
        assert math.isclose(bern.log_prob(flips).sum().item(), log_likelihood(0.7, FLIPS), rel_tol=1e-12)

        z = np.array([2.0, -1.0, 0.5, 0.0])                      # 4 個 token 的分數
        tokens = np.array([0, 2, 2, 3, 1, 0])                    # 一串 token 的 ID
        p = softmax(z)                                           # EP01
        torch_lp = torch.log_softmax(torch.tensor(z), dim=-1)[torch.tensor(tokens)].sum().item()
        assert math.isclose(chain_log_prob(p[tokens]), torch_lp, rel_tol=1e-12)

    tests = [empirical_distribution_counts_and_sums_to_one, empirical_distribution_approaches_true_distribution,
             conditional_rows_sum_to_one_and_rebuild_joint, conditional_zero_row_raises,
             chain_log_prob_no_underflow, bernoulli_mle_maximizes_log_likelihood,
             log_likelihood_edge_cases, matches_pytorch]
    passed = skipped = pending = 0
    for t in tests:
        try:
            t()
            print("✓", t.__name__)
            passed += 1
        except NotImplementedError as e:
            print("…", t.__name__, f"（{e}）")
            pending += 1
        except Skipped as e:
            print("-", t.__name__, f"跳過（{e}）")
            skipped += 1
        except Exception as e:
            msg = f"{type(e).__name__}: {e}" if str(e) else type(e).__name__
            frames = [fr for fr in traceback.extract_tb(e.__traceback__) if fr.name == t.__name__]
            line = frames[-1].line if frames else ""                    # 檢查裡出錯的那一行
            print("✗", t.__name__, msg, f"｜{line}" if line else "")
    summary = f"{passed}/{len(tests)} 通過"
    if skipped:
        summary += f"，{skipped} 個跳過"
    if pending:
        summary += f"，{pending} 個還沒寫好：先完成上一格，或執行最後的參考解答"
    print(summary)


check_prob()

### 畫出概似函數：`draw_likelihood`、`draw_log_likelihood`

網站上這是一個畫圖腳本 `llm-labs/ep06/plot_likelihood.py`，執行後存成 `ep06/likelihood.png`；在 notebook 裡只要在下一格寫好 `draw_likelihood`、`draw_log_likelihood` 這 2 個函式，再下一格會用它們畫一列兩張圖（左邊 `draw_likelihood`、右邊 `draw_log_likelihood`），直接顯示。`FLIPS` 是工具四的 10 次。

要求與提示：

1. **兩張圖**：左圖的標題寫出式子，例如 `p^7 (1 - p)^3`（用正面數、反面數組出來）。右圖的兩端是 $\log 0$，所以只畫 0.01 到 0.99，`ylim=(-35, 0)`。
2. **標出最大值**：寫一個兩張圖共用的小函式 `mark_max(ax, x, y, text)`，用 `ax.plot([x], [y], "o", color="red")`、`ax.axvline(x, color="red", linestyle="--")`、`ax.annotate(text, (x, y))`。
3. 這兩個函式自己算 $L$ 和 $\log L$，不呼叫上面的 `log_likelihood`：網站上的腳本是直接執行的，Python 只在 `ep06/` 裡找模組，找不到 `ep06.prob`。

In [ ]:
FLIPS = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 0])     # 工具四的 10 次：7 次正面


def draw_likelihood(ax, flips):
    """在 ax 上畫 L(p) = p^正面數 (1 − p)^反面數，p 從 0 到 1（501 個點）；
    在 p̂ 畫紅點和紅色垂直虛線，標出 p̂ 和 L(p̂)，標題寫出式子。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw_likelihood")


def draw_log_likelihood(ax, flips):
    """在 ax 上畫 log L(p) = 正面數 · ln p + 反面數 · ln(1 − p)，p 從 0.01 到 0.99；
    同樣標出 p̂ 和 log L(p̂)。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw_log_likelihood")

下一格用你的 2 個函式畫圖，再檢查畫出來的東西。畫對的話就是工具四說的那張圖：左圖的山頂在 0.7、高約 0.00222；右圖在 0.7 最高（約 −6.11），$p = 0.01$ 約 −32、$p = 0.99$ 約 −14。最後一個檢查換一份資料（5 次裡 2 次正面）再畫一次（不顯示），確認標題和 $\hat{p}$ 是用正面數、反面數算出來的，不是寫死的。

In [ ]:
from matplotlib.colors import same_color


def show_likelihood_plot():
    """一列兩張圖：左邊 draw_likelihood、右邊 draw_log_likelihood，再檢查曲線、紅點、紅色虛線、標示、標題和範圍。"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
    try:
        draw_likelihood(ax1, FLIPS)
        draw_log_likelihood(ax2, FLIPS)
    except NotImplementedError as e:
        plt.close(fig)                  # 不顯示空白的圖
        print(f"{e}：先完成上一格，或執行最後的參考解答，再回來執行這格")
        return

    def L(p, heads=7, tails=3):                              # 檢查自己算一份
        return p ** heads * (1 - p) ** tails

    def logL(p, heads=7, tails=3):
        return heads * np.log(p) + tails * np.log(1 - p)

    def lines(ax):                                           # ax 上的每條線，排成 (2, N)
        return [np.array(l.get_data(), dtype=float) for l in ax.get_lines()]

    def curve(ax, f, lo, hi, n=None):                        # 有沒有一條線：x 從 lo 到 hi（n 個點），y 是 f(x)
        return any(d.shape[1] >= 50 and (n is None or d.shape[1] == n)
                   and np.isclose(d[0].min(), lo) and np.isclose(d[0].max(), hi)
                   and np.allclose(d[1], f(d[0]), rtol=1e-9, atol=1e-12) for d in lines(ax))

    def red_dot(ax, x, y):                                   # 在 (x, y) 有沒有一個紅點
        return any(l.get_marker() == "o" and same_color(l.get_color(), "red") and len(l.get_xdata()) == 1
                   and np.allclose([l.get_xdata()[0], l.get_ydata()[0]], [x, y]) for l in ax.get_lines())

    def red_vline(ax, x):                                    # 在 x 有沒有一條紅色垂直虛線
        return any(l.get_linestyle() == "--" and same_color(l.get_color(), "red")
                   and np.allclose(np.asarray(l.get_xdata(), dtype=float), x) for l in ax.get_lines())

    def marked(ax, x, y):                                    # 有沒有用 annotate 在 (x, y) 寫出 p̂
        return any(hasattr(t, "xy") and np.allclose(t.xy, (x, y)) and f"{x:g}" in t.get_text() for t in ax.texts)

    def short_lines(ax):                                     # 點數很少的線（紅點、垂直線）：線型、顏色、x
        return [(l.get_linestyle(), l.get_marker(), l.get_color(), [round(float(v), 4) for v in l.get_xdata()])
                for l in ax.get_lines() if len(l.get_xdata()) <= 2]

    def no_space(s):
        return s.replace(" ", "")

    other = {}

    def draw_other():                                        # 換一份資料（5 次裡 2 次正面）再畫一次，不顯示
        if not other:
            fig2, (b1, b2) = plt.subplots(1, 2)
            try:
                draw_likelihood(b1, np.array([1, 0, 0, 1, 0]))
                draw_log_likelihood(b2, np.array([1, 0, 0, 1, 0]))
            finally:
                plt.close(fig2)
            other.update(b1=b1, b2=b2)
        return other["b1"], other["b2"]

    checks = [
        ("左圖：L(p) = p^7 (1 - p)^3 的曲線，p 從 0 到 1（501 個點）",
         lambda: curve(ax1, L, 0, 1, n=501), lambda: f"每條線的點數 {[d.shape[1] for d in lines(ax1)]}"),
        ("左圖：在 p̂ = 0.7 有紅點和紅色垂直虛線",
         lambda: red_dot(ax1, 0.7, L(0.7)) and red_vline(ax1, 0.7), lambda: short_lines(ax1)),
        ("左圖：用 annotate 在紅點旁寫出 p̂",
         lambda: marked(ax1, 0.7, L(0.7)), lambda: [(getattr(t, "xy", None), t.get_text()) for t in ax1.texts]),
        ("左圖：標題寫出式子 p^7 (1 - p)^3",
         lambda: "p^7(1-p)^3" in no_space(ax1.get_title()), lambda: repr(ax1.get_title())),
        ("右圖：log L(p) = 7 ln p + 3 ln(1 - p) 的曲線，p 從 0.01 到 0.99",
         lambda: curve(ax2, logL, 0.01, 0.99),
         lambda: f"每條線的（點數, 最小 p, 最大 p）{[(d.shape[1], round(float(d[0].min()), 4), round(float(d[0].max()), 4)) for d in lines(ax2)]}"),
        ("右圖：ylim=(-35, 0)", lambda: np.allclose(ax2.get_ylim(), (-35, 0)),
         lambda: tuple(round(float(v), 2) for v in ax2.get_ylim())),
        ("右圖：在 p̂ = 0.7 有紅點和紅色垂直虛線",
         lambda: red_dot(ax2, 0.7, logL(0.7)) and red_vline(ax2, 0.7), lambda: short_lines(ax2)),
        ("右圖：用 annotate 在紅點旁寫出 p̂",
         lambda: marked(ax2, 0.7, logL(0.7)), lambda: [(getattr(t, "xy", None), t.get_text()) for t in ax2.texts]),
        ("換一份資料（5 次裡 2 次正面）：標題 p^2 (1 - p)^3，兩張圖的紅點都在 p̂ = 0.4",
         lambda: "p^2(1-p)^3" in no_space(draw_other()[0].get_title())
                 and red_dot(draw_other()[0], 0.4, L(0.4, 2, 3)) and red_dot(draw_other()[1], 0.4, logL(0.4, 2, 3)),
         lambda: (repr(draw_other()[0].get_title()), short_lines(draw_other()[0]), short_lines(draw_other()[1]))),
    ]
    plt.show()
    for name, ok, got in checks:
        try:
            passed = bool(ok())
        except Exception as e:                               # 例如畫圖時出錯，連比都沒辦法比
            passed, got = False, (lambda e=e: f"{type(e).__name__}: {e}")
        if passed:
            print("✓", name)
        else:
            try:
                print("✗", f"{name}：現在是 {got()}")
            except Exception as e:
                print("✗", f"{name}：{type(e).__name__}: {e}")


show_likelihood_plot()

網站上的腳本把主程式放在 `if __name__ == "__main__":` 底下，用 `matplotlib.use("Agg")` 只存檔、不開視窗，並寫死 `figsize` 和 `dpi`，執行兩次的 `md5sum` 指紋一樣（同 EP04、EP05）；在 notebook 裡用 `plt.show()` 直接顯示就好。

### 和 PyTorch 對照

用你在上面寫的 `log_likelihood`、`chain_log_prob`，和 PyTorch 對照（Colab 已經內建 PyTorch；沒裝的話這格會跳過）。會用到前面的 `flips`、`data`、`z`、`log_lik_z`。第一行是工具四那份資料的 log 概似；第二行是玩具模型走完之後，那 10 個 token 的 log 機率總和；第三行是玩具模型起點的梯度，左邊用 PyTorch 的 `.backward()`，右邊用數值梯度：

In [ ]:
try:
    import torch

    bern = torch.distributions.Bernoulli(probs=torch.tensor(0.7, dtype=torch.float64))
    lp = bern.log_prob(torch.tensor(flips, dtype=torch.float64))   # 每一次丟的 log 機率
    print(lp.sum().item(), log_likelihood(0.7, flips))

    z_t = torch.tensor(z)                                          # 玩具模型走完的分數，float64
    print(torch.log_softmax(z_t, dim=-1)[torch.tensor(data)].sum().item(),
          chain_log_prob(softmax(z)[data]))

    z0 = torch.zeros(3, dtype=torch.float64, requires_grad=True)   # 玩具模型的起點
    torch.log_softmax(z0, dim=-1)[torch.tensor(data)].sum().backward()   # 跑一次、倒著算回來（EP05）
    print(z0.grad, numerical_gradient(log_lik_z, np.zeros(3)))
except ImportError:
    print("沒有安裝 PyTorch：跳過這格")
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

`torch.distributions` 的分佈都有 `log_prob`：回傳每一筆資料的 log 機率，加起來就是 log 概似。LLM 訓練時算的就是第二行這種「log-softmax 取出真正的 token、加起來」，梯度用 `.backward()`，和數值梯度一樣（PyTorch 只印 4 位小數）。

## 7. 弄壞它

每一個都先**猜**結果，再執行，最後用自己的話解釋。完整的解釋在網站的「弄壞它」段落。

弄壞它 1 用到工具二的 `joint`，弄壞它 3 用到工具四的 `flips`、`ps`，弄壞它 4 用到第 6 節的 `bernoulli_mle`、`log_likelihood`。重新啟動過 kernel 的話，先重新執行第一格、定義前幾集函式的那一格，以及定義這些變數的格子。

### 弄壞它 1：條件機率除以錯的總和

除以整張表的總和、除以每一欄的總和、忘了 `keepdims`，每一列加起來還是 1 嗎？第一行是前兩種做法每一列的總和，以及除以欄的總和時每一欄的總和；接著印出除以欄的總和得到的表。（最後一行會出錯：這裡用 `try`／`except` 接住錯誤、只印出訊息，notebook 才不會停下來。）

In [ ]:
wrong_total = joint / joint.sum()                # 除以整張表的總和
wrong_col = joint / joint.sum(axis=0)            # 除以每一欄的總和
print(wrong_total.sum(axis=1), wrong_col.sum(axis=1), wrong_col.sum(axis=0))
print(np.round(wrong_col, 3))
try:
    joint / joint.sum(axis=1)                    # 忘了 keepdims
except ValueError as e:
    print("ValueError:", e)

- 整張表的總和本來就是 1，除了等於沒除：那還是聯合機率，每一列加起來是 $P(A)$，不是 1。**聯合機率不是條件機率**。
- 除以欄的總和，變成每一**欄**加起來是 1：算出來的是 $P(A \mid B)$，方向反了。左上角 0.8 是 $P(\text{很} \mid \text{冷})$（下一個字是「冷」時，前一個字是「很」的機率），不是 $P(\text{冷} \mid \text{很}) = 0.333$。
- 忘了 `keepdims`：`(2, 3)` 除以 `(2,)`，廣播從最後一維對齊，3 和 2 對不上，NumPy 擋了下來（EP01）。**方陣就擋不下來**：3 × 3 的表除以 `(3,)`，第 $j$ 欄會除以第 $j$ 列的總和，照樣算出錯的數字（所以檢查用了一張 3 × 3 的表）。

### 弄壞它 2：把 token 當成彼此獨立

每個 token 單獨出現的機率（示範數字）直接連乘，「今天天氣很冷」和「冷很氣天今天」誰的機率大？（左邊是正常的順序，右邊是倒過來）

In [ ]:
p_single = {"今天": 0.002, "天": 0.004, "氣": 0.003, "很": 0.01, "冷": 0.001}   # 示範
forward = ["今天", "天", "氣", "很", "冷"]
backward = ["冷", "很", "氣", "天", "今天"]
print(np.prod([p_single[t] for t in forward]), np.prod([p_single[t] for t in backward]))

一模一樣（最後一位是捨入誤差）。乘法可以交換順序，所以「當成獨立、直接相乘」完全看不到順序。換成條件機率，「冷」開頭以後，$P(\text{很} \mid \text{冷})$、$P(\text{氣} \mid \text{冷很})$ 都非常小，順序就回來了。只要每一步的條件機率是對的，鏈鎖律保證連乘起來就是整句話的機率，不用假設任何獨立。

### 弄壞它 3：連乘下溢，概似找不到山頂

1000 個 0.1 直接連乘；工具四的 10 次重複 1000 遍（一萬次：7000 正、3000 反），用格點找概似最大的 $p$。第一行是 1000 個 0.1 的連乘和 log 加總；第二行依序是概似格點裡的最大值、它的位置，以及 log 概似的最大位置和最大值：

In [ ]:
print(np.prod(np.full(1000, 0.1)), np.sum(np.log(np.full(1000, 0.1))))
big = np.tile(flips, 1000)                       # np.tile：整串重複 1000 遍
L_big = np.array([np.prod(np.where(big == 1, p, 1 - p)) for p in ps])
inner = ps[1:-1]                                 # 去掉 0 和 1：log 0 是 -inf
ll_big = 7000 * np.log(inner) + 3000 * np.log(1 - inner)
print(L_big.max(), ps[np.argmax(L_big)], inner[np.argmax(ll_big)], ll_big.max())

$0.1^{1000} = 10^{-1000}$ 遠小於 `float64` 能存的最小正數（約 $5 \times 10^{-324}$，EP02），下溢成 0；log 加總是很普通的 $-2302.59$。一萬次丟硬幣的 $L(0.7) = e^{-6108.6}$ 也下溢成 0：101 個格點**全部是 0**，`argmax` 只好回傳第一個位置，說「最大概似估計是 $p = 0$」，安安靜靜地錯了。$\log L$ 沒問題，找到 0.7。訓練資料有上兆個 token，所以一定要算 log 概似。

### 弄壞它 4：資料太少的最大概似估計

丟 3 次都是正面，$\hat{p}$ 是多少？第 4 次出現反面，這 4 次的 log 概似是多少？（用你在第 6 節寫的 `bernoulli_mle`、`log_likelihood`；印出 $\hat{p}$、這 4 次的 log 概似，以及直接算的 $\ln(1 - \hat{p})$。最後一個是 $\ln 0$，會跳出 `RuntimeWarning`：那只是警告，不會讓 notebook 停下來。）

In [ ]:
few = np.array([1, 1, 1])
try:
    p_hat = bernoulli_mle(few)
    print(p_hat, log_likelihood(p_hat, [1, 1, 1, 0]), np.log(1 - p_hat))   # 最後一個是 ln 0：會跳出 RuntimeWarning
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

`1.0 -inf -inf`：`log_likelihood` 處理了 $\ln 0$，所以沒有警告；直接算的 `np.log(1 - p_hat)` 跳出 `RuntimeWarning: divide by zero encountered in log`。只看到 3 次正面，最大概似估計就說「反面不可能」；第 4 次一出現反面，整份資料的機率是 0，log 是 $-\infty$。**資料裡沒出現過的事，最大概似估計給 0**。用計數做的語言模型也一樣：訓練資料裡沒出現過的「兩個字的組合」機率是 0，一句話只要有一個這種組合，整句的機率就是 0；EP17 會用**平滑 smoothing** 解決。

### 弄壞它 5：沒有固定 seed

同一段模擬跑三次（每一行是一次），左欄沒給 seed、右欄用 seed 0，印出丟 100 次正面的比例。**先猜猜看**：三行的左欄會一樣嗎？右欄呢？

In [ ]:
for run in range(3):
    print(np.random.default_rng().integers(0, 2, size=100).mean(),    # 沒給 seed
          np.random.default_rng(0).integers(0, 2, size=100).mean())   # seed 0

**這一格的左欄每次執行都不一樣**：沒給 seed，每次都用新的亂數，三行多半不同；你跑出來的也會和網站上寫的（某一次跑出 0.45、0.52、0.46）不一樣，再執行一次這格又會變。右欄永遠是 0.56（工具一前 100 次的比例）。沒固定 seed，換了設定之後結果變了，就分不出是設定的影響還是運氣；固定 seed，實驗才能重現（EP01）。

## 下一集

這集知道了訓練語言模型的目標：讓訓練資料的 log 機率總和越大越好。下一集 **EP07 資訊理論：Softmax、熵、交叉熵、KL 散度、困惑度** 把這個總和取負號、再除以 token 數，變成 LLM 訓練時的損失，並說明它為什麼叫交叉熵，熵、KL 散度、困惑度又各量什麼。

## 參考解答

先自己寫，卡住再看。執行下一格會用參考解答取代第 6 節的 5 個函式，並重新跑一次檢查；再下一格是 `draw_likelihood`、`draw_log_likelihood` 的參考解答，會畫出工具四那張概似函數的圖。之後可以回到上面，重新執行「和 PyTorch 對照」和弄壞它 4 那兩格。

In [ ]:
# 只用 NumPy 和基本的 Python。
import numpy as np


def empirical_distribution(samples, k):
    """samples：一串整數，每個都在 0 到 k − 1（k 是類別數，工具四的 K）。回傳長度 k 的陣列，第 i 格是 i 出現的比例
    （沒出現就是 0），加起來是 1。samples 是空的、有值不在 0 到 k − 1、或不是整數，要 raise ValueError。"""
    samples = np.asarray(samples)
    if samples.size == 0:
        raise ValueError("samples 是空的，沒有比例可算")
    if not np.issubdtype(samples.dtype, np.integer):
        raise ValueError(f"samples 要是整數，收到 {samples.dtype}")
    if samples.min() < 0 or samples.max() >= k:
        raise ValueError(f"samples 的值要在 0 到 {k - 1} 之間")
    counts = np.bincount(samples.ravel(), minlength=k)
    return counts / samples.size


def conditional(joint):
    """joint：二維聯合機率表，第 i 列第 j 欄是 P(A = i, B = j)。回傳同形狀的條件機率表，
    第 i 列是 P(B | A = i)：每一列除以那一列的加總 P(A = i)，所以每一列加起來是 1。
    某一列加總是 0（P(A = i) = 0，條件機率沒有定義）要 raise ValueError。不能改到呼叫端的 joint。"""
    joint = np.asarray(joint, dtype=float)
    if joint.ndim != 2:
        raise ValueError(f"joint 要是二維的表，收到形狀 {joint.shape}")
    row_sums = joint.sum(axis=1, keepdims=True)        # (m, 1)：每一列自己的 P(A = i)
    if np.any(row_sums == 0):
        bad = np.flatnonzero(row_sums[:, 0] == 0)
        raise ValueError(f"第 {bad.tolist()} 列加總是 0，條件機率沒有定義")
    return joint / row_sums                            # (m, n) / (m, 1)：每一列除以自己的加總


def chain_log_prob(cond_probs):
    """一串條件機率 P(x₁)、P(x₂ | x₁)、⋯、P(xₙ | x₁, ⋯, xₙ₋₁) → log P(x₁, ⋯, xₙ) = Σ log。
    不先連乘（長序列不能下溢）。每個機率都要在 0 到 1 之間，否則 raise ValueError；
    有 0 時回傳 -inf，而且不能跳出 RuntimeWarning。"""
    p = np.asarray(cond_probs, dtype=float)
    if np.any(np.isnan(p)) or np.any(p < 0) or np.any(p > 1):
        raise ValueError("機率要在 0 到 1 之間")
    if np.any(p == 0):
        return -np.inf                                 # 有一步不可能 → 整句不可能
    return float(sequence_logprob(p))                  # EP02：Σ log pᵢ（最上面已經定義）


def bernoulli_mle(flips):
    """flips：0/1 陣列（1 = 正面）。回傳正面的比例，也就是白努利分佈的最大概似估計 p̂。
    空的、或有 0 和 1 以外的值，要 raise ValueError。"""
    flips = np.asarray(flips)
    if flips.size == 0:
        raise ValueError("flips 是空的")
    if not np.all((flips == 0) | (flips == 1)):
        raise ValueError("flips 只能有 0 和 1")
    return float(flips.mean())


def log_likelihood(p, flips):
    """白努利分佈的 log 概似：(正面數) ln p + (反面數) ln(1 − p)。
    p 可以是 0 或 1，0 × log 0 當成 0：p = 1 而且全部正面是 0.0、p = 1 卻有反面是 -inf（p = 0 對稱）；
    不能跳出 RuntimeWarning。p 不在 [0, 1] 要 raise ValueError。"""
    flips = np.asarray(flips)
    if not 0 <= p <= 1:
        raise ValueError(f"p 要在 0 到 1 之間，收到 {p}")
    heads = int(np.sum(flips == 1))
    tails = int(flips.size - heads)

    def term(count, prob):                             # count × ln(prob)，0 × log 0 規定是 0
        if count == 0:
            return 0.0
        if prob == 0:
            return -np.inf
        return count * np.log(prob)

    return float(term(heads, p) + term(tails, 1 - p))


check_prob()

In [ ]:
FLIPS = np.array([1, 1, 0, 1, 1, 1, 0, 1, 1, 0])     # 工具四的 10 次：7 次正面


def mark_max(ax, x, y, text, offset=(10, -40)):
    """在 (x, y) 畫一個紅點和一條垂直虛線，旁邊寫 text（offset：文字離紅點幾點，右、上為正）。"""
    ax.axvline(x, color="red", linestyle="--", linewidth=1)
    ax.plot([x], [y], "o", color="red")
    ax.annotate(text, (x, y), xytext=offset, textcoords="offset points", color="red")


def draw_likelihood(ax, flips):
    """在 ax 上畫 L(p) = p^正面數 (1 − p)^反面數，p 從 0 到 1（501 個點）；
    在 p̂ 畫紅點和紅色垂直虛線，標出 p̂ 和 L(p̂)，標題寫出式子。"""
    heads, tails = int(flips.sum()), int(flips.size - flips.sum())
    ps = np.linspace(0, 1, 501)
    p_hat = flips.mean()
    ax.plot(ps, ps ** heads * (1 - ps) ** tails, color="C0", linewidth=2)
    L_max = p_hat ** heads * (1 - p_hat) ** tails
    mark_max(ax, p_hat, L_max, f"p̂ = {p_hat:g}\nL = {L_max:.5f}", offset=(-150, -10))
    ax.set(xlim=(0, 1), ylim=(0, None), xlabel="p (probability of heads)", ylabel="L(p)",
           title=f"likelihood L(p) = p^{heads} (1 - p)^{tails}")
    ax.grid(True)


def draw_log_likelihood(ax, flips):
    """在 ax 上畫 log L(p) = 正面數 · ln p + 反面數 · ln(1 − p)，p 從 0.01 到 0.99；
    同樣標出 p̂ 和 log L(p̂)。"""
    heads, tails = int(flips.sum()), int(flips.size - flips.sum())
    ps = np.linspace(0.01, 0.99, 491)                  # 兩端是 log 0 = -inf，只畫 0.01 到 0.99
    p_hat = flips.mean()
    ax.plot(ps, heads * np.log(ps) + tails * np.log(1 - ps), color="C1", linewidth=2)
    ll_max = heads * np.log(p_hat) + tails * np.log(1 - p_hat)
    mark_max(ax, p_hat, ll_max, f"p̂ = {p_hat:g}\nlog L = {ll_max:.2f}")
    ax.set(xlim=(0, 1), ylim=(-35, 0), xlabel="p (probability of heads)", ylabel="log L(p)",
           title=f"log likelihood {heads} ln p + {tails} ln(1 - p)")
    ax.grid(True)


show_likelihood_plot()